# DINOv2 + LSTM-MHA for blastocyst formation prediction from sparse daily embryo images

Trains and evaluates the LSTM-MHA model and the LSTM baseline for the input windows Day 0-2 (W48), Day 0-3 (W72)
and Day 0-6 (W144) with 10 stratified 80/20 splits and 5-fold inner cross-validation, together with a
logistic-regression baseline, and writes all tables and figures of the paper.

**Input:** `Classes.xlsx` (columns `video_name`, `Class`) and `Embedding/` with one parquet file per embryo
(rows = daily frames, 1536-d DINOv2 ViT-B/14 descriptors: [CLS] + mean patch token).
**Run:** Google Colab, GPU runtime, Runtime > Run all. Finished runs are saved and skipped when the notebook is run again.

In [ ]:
# ============================ 1. Configuration ============================
import os, re, json, time, pickle, platform, gc, warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import tensorflow as tf, keras, psutil, glob, logging
tf.get_logger().setLevel(logging.ERROR)
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
from keras import layers
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, accuracy_score,
                             precision_recall_fscore_support, confusion_matrix, roc_curve)
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon

try:
    from google.colab import drive
    drive.mount("/content/drive"); BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = "."

LABELS_PATH = f"{BASE}/Classes.xlsx"
EMB_DIR     = f"{BASE}/Embedding/"
OUT_DIR     = f"{BASE}/Revision_results/"

PROFILE = "standard"            # "quick" (smoke test, ~5 min) | "standard" (all results for the paper)
RUN_WINDOWS = ["W48", "W72", "W144"]
REPRESENTATION = "auto"         # "auto" = raw / ref / ref_pca chosen per split on training data (same as the runs already done)
SAVE_WEIGHTS = "split0"         # "none" | "split0" (~0.5 GB) | "all" (several GB of Drive space!)
CLEAN_OLD_WEIGHTS = False       # delete weight files of earlier runs except split 0 of LSTM-MHA (frees Drive space)

N_SPLITS    = {"quick": 2, "standard": 10}[PROFILE]
INNER_FOLDS = {"quick": 2, "standard": 5}[PROFILE]
EPOCHS      = {"quick": 3, "standard": 50}[PROFILE]
TEST_FRAC   = 0.20
BATCH       = 32
LR          = 1e-3
PATIENCE    = 10                # EarlyStopping on the inner-fold validation loss
LR_PATIENCE = 4                 # ReduceLROnPlateau (must be smaller than PATIENCE to have an effect)
FRAME_HOURS = [0, 24, 48, 72, 96, 120, 144, 168]
WINDOWS = {"W48": 3, "W72": 4, "W144": 7}                     # number of daily frames (0 h ... cutoff)

# Input representations of the DINOv2 sequence (same network for all; chosen per split on training data only)
#   raw      : standardized DINOv2 descriptors (as in the paper)
#   ref      : change of each frame relative to the embryo's own Day-0 frame
#   ref_pca  : 'ref' reduced to K principal components (PCA fitted on training frames only)
ALL_REPS = {"raw": dict(name="raw", pca=None), "ref": dict(name="ref", pca=None), "ref_pca": dict(name="ref_pca", pca=32)}
REPRESENTATIONS = list(ALL_REPS.values()) if REPRESENTATION == "auto" else [ALL_REPS[REPRESENTATION]]
D = 1536

os.makedirs(f"{OUT_DIR}/runs", exist_ok=True); os.makedirs(f"{OUT_DIR}/weights", exist_ok=True)

# --- reuse results of earlier runs (older notebook saved them as ..._r0/_r1/_r2.pkl) ---
_old = {"r0": "raw", "r1": "ref", "r2": "ref_pca"}
for f in glob.glob(f"{OUT_DIR}/runs/*_r[012].pkl"):
    new = f"{f[:-7]}_{_old[f[-6:-4]]}.pkl"
    if not os.path.exists(new): os.rename(f, new)
print("finished runs found:", len(glob.glob(f"{OUT_DIR}/runs/*.pkl")))
# --- free Drive space: old version saved the weights of every fold ---
if CLEAN_OLD_WEIGHTS:
    freed = 0
    for f in glob.glob(f"{OUT_DIR}/weights/*.weights.h5"):
        if "LSTM-MHA_s0_" not in os.path.basename(f):
            freed += os.path.getsize(f); os.remove(f)
    if freed: print(f"deleted old weight files: {freed/1e9:.1f} GB (also empty the Google Drive trash once)")
env = dict(python=platform.python_version(), tensorflow=tf.__version__, keras=keras.__version__,
           sklearn=sklearn.__version__, numpy=np.__version__, gpu=str(tf.config.list_physical_devices("GPU")))
json.dump(dict(env=env, PROFILE=PROFILE, REPRESENTATION=REPRESENTATION, N_SPLITS=N_SPLITS, INNER_FOLDS=INNER_FOLDS, EPOCHS=EPOCHS, BATCH=BATCH, LR=LR,
               PATIENCE=PATIENCE, LR_PATIENCE=LR_PATIENCE, TEST_FRAC=TEST_FRAC, WINDOWS=WINDOWS,
               REPRESENTATIONS=REPRESENTATIONS), open(f"{OUT_DIR}/config.json", "w"), indent=2)
_todo = sum(not os.path.exists(f"{OUT_DIR}/runs/{w}_{m}_s{s_}_{rp['name']}.pkl")
            for w in RUN_WINDOWS for m in ["LSTM-MHA", "LSTM"] for s_ in range(N_SPLITS) for rp in REPRESENTATIONS)
print(env); print(f"remaining runs: {_todo}  (= {_todo*INNER_FOLDS} trainings, GPU roughly {_todo*INNER_FOLDS*6/3600:.1f}-{_todo*INNER_FOLDS*12/3600:.1f} h)")
print(f"Free disk on Drive/OUT_DIR: {psutil.disk_usage(OUT_DIR).free/1e9:.1f} GB, RAM used: {psutil.virtual_memory().percent:.0f}%")

In [ ]:
# ============================ 2. Data ============================
def norm_name(s):
    return re.sub(r"[_\-\s]", "", str(s).lower().strip())

lab = pd.read_excel(LABELS_PATH)
lab_map = {norm_name(v): (v, c) for v, c in zip(lab["video_name"], lab["Class"])}
names, seqs, y, unmatched = [], [], [], []
for f in sorted(os.listdir(EMB_DIR)):
    if not f.endswith(".parquet"): continue
    key = norm_name(re.sub(r"your_out_path_for_df_|\.parquet", "", f))
    if key not in lab_map:
        unmatched.append(f); continue
    e = pd.read_parquet(os.path.join(EMB_DIR, f)).values.astype("float32")
    names.append(lab_map[key][0]); seqs.append(e); y.append(int(lab_map[key][1] == "Blasto"))
names, y = np.array(names), np.array(y)
n_rows = np.array([len(s) for s in seqs]); T_MAX = int(n_rows.max())
X_raw = np.zeros((len(seqs), T_MAX, D), "float32")
for i, s in enumerate(seqs): X_raw[i, :len(s)] = s

print(f"{len(y)} embryos: Blastocyst={y.sum()}, non-Blastocyst={len(y)-y.sum()}, unmatched files={len(unmatched)}")
if unmatched: print("unmatched:", unmatched[:20])
fpc = pd.crosstab(pd.Series(n_rows, name="frames"), pd.Series(np.where(y == 1, "Blastocyst", "non-Blastocyst"), name="class"))
fpc.to_csv(f"{OUT_DIR}/frames_per_class.csv"); print(fpc)

In [ ]:
# ============================ 3. Splits ============================
splits = []
for s_ in range(N_SPLITS):
    tr, te = train_test_split(np.arange(len(y)), test_size=TEST_FRAC, stratify=y, random_state=s_)
    tr = np.sort(tr)
    skf = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=s_)
    folds = [(tr[a].tolist(), tr[b].tolist()) for a, b in skf.split(tr, y[tr])]
    splits.append(dict(seed=s_, train=tr.tolist(), test=np.sort(te).tolist(), folds=folds))
json.dump(dict(names=names.tolist(), labels=y.tolist(), splits=splits), open(f"{OUT_DIR}/splits.json", "w"))
print(f"train={len(splits[0]['train'])}  test={len(splits[0]['test'])}  inner folds={INNER_FOLDS}")

In [ ]:
# ============================ 4. Input representation (all statistics from training embryos only) ============================
def real_mask(X):
    return np.abs(X).sum(-1) > 0

def prepare(Xw, fit_idx, rep, seed):
    m = real_mask(Xw)
    F = Xw[fit_idx][m[fit_idx]]
    mu, sd = F.mean(0), F.std(0) + 1e-6
    Z = np.zeros_like(Xw); Z[m] = (Xw[m] - mu) / sd                       # standardized, padding stays 0
    if rep["name"] in ("ref", "ref_pca"):
        R = (Z - Z[:, :1]) * m[..., None]                                   # change relative to the embryo's Day-0 frame
        R[:, 0] = Z[:, 0]                                                   # Day-0 frame itself is kept
        Z = R
    if rep["pca"]:
        k = rep["pca"]; fit_rows = Z[fit_idx][:, 1:][m[fit_idx][:, 1:]]
        pca = PCA(k, whiten=True, random_state=seed).fit(fit_rows)
        P = np.zeros(Z.shape[:2] + (k,), "float32"); P[m] = pca.transform(Z[m]); Z = P
    return Z.astype("float32")

In [ ]:
# ============================ 5. Model (architecture of the paper) ============================
def build_model(T, in_dim, use_mha=True, seed=0):
    keras.utils.set_random_seed(seed)
    inp = keras.Input(shape=(T, in_dim))
    x = layers.LSTM(256, return_sequences=True, dropout=0.2)(inp)
    x = layers.LayerNormalization()(x); x = layers.Dropout(0.2)(x)
    x = layers.LSTM(128, return_sequences=True, dropout=0.2)(x)
    x = layers.LayerNormalization()(x); x = layers.Dropout(0.2)(x)
    scores = None
    if use_mha:
        att, scores = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x, return_attention_scores=True)
        x = layers.LayerNormalization()(x + att)
    x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(LR), loss="binary_crossentropy", metrics=["accuracy"])
    att_model = keras.Model(inp, scores) if use_mha else None
    return model, att_model

def temporal_macs(T, in_dim, use_mha):
    m = T * 4 * 256 * (in_dim + 256) + T * 4 * 128 * (256 + 128)
    if use_mha: m += 4 * T * 128 * 256 + 2 * T * T * 256
    return m + T * 128

for name, d_ in [("raw/ref", D), ("ref_pca", 32)]:
    print(name, "params MHA:", build_model(7, d_, True)[0].count_params(), " LSTM only:", build_model(7, d_, False)[0].count_params())

In [ ]:
# ============================ 6. Training and evaluation ============================
def metrics(yt, prob, thr=0.5):
    yp = (prob >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(yt, yp, labels=[0, 1], zero_division=0)
    return dict(auc=roc_auc_score(yt, prob), bal_acc=100 * balanced_accuracy_score(yt, yp),
                acc=100 * accuracy_score(yt, yp), prec_non=p[0], rec_non=r[0], f1_non=f[0],
                prec_bl=p[1], rec_bl=r[1], f1_bl=f[1])

def best_threshold(yt, prob):
    cand = np.unique(np.concatenate([[0.5], prob]))
    return float(cand[int(np.argmax([balanced_accuracy_score(yt, (prob >= t).astype(int)) for t in cand]))])

def run_rep(win, T, model_name, sp, ri, rep):
    path = f"{OUT_DIR}/runs/{win}_{model_name}_s{sp['seed']}_{rep['name']}.pkl"
    if os.path.exists(path):
        try: return pickle.load(open(path, "rb"))
        except Exception: os.remove(path)                                   # incomplete file from a disconnect -> redo
    if psutil.virtual_memory().percent > 85:
        raise RuntimeError("RAM almost full. Runtime -> Restart session -> Run all (finished runs are kept).")
    use_mha = model_name == "LSTM-MHA"; te = np.array(sp["test"]); Xw = X_raw[:, :T]
    oof_i, oof_p, test_p, att, ep, bep, hist0, tmin = [], [], [], [], [], [], None, 0.0
    for k, (itr, iva) in enumerate(sp["folds"]):
        itr, iva = np.array(itr), np.array(iva)
        Z = prepare(Xw, itr, rep, sp["seed"])
        keras.backend.clear_session(); gc.collect()
        model, att_model = build_model(T, Z.shape[-1], use_mha, seed=1000 * sp["seed"] + 10 * k + ri)
        cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y[itr])
        cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=LR_PATIENCE, min_lr=1e-5)]
        t0 = time.time()
        h = model.fit(Z[itr], y[itr], validation_data=(Z[iva], y[iva]), epochs=EPOCHS, batch_size=BATCH,
                      class_weight={0: cw[0], 1: cw[1]}, callbacks=cb, verbose=0)
        tmin += (time.time() - t0) / 60
        oof_i.append(iva); oof_p.append(np.asarray(model(Z[iva], training=False)).ravel())
        test_p.append(np.asarray(model(Z[te], training=False)).ravel())
        if use_mha:
            a = np.asarray(att_model(Z[te], training=False))                   # (n, heads, T, T)
            att.append(a.mean(axis=(1, 2)))                                      # attention received per frame
        ep.append(len(h.history["loss"])); bep.append(int(np.argmin(h.history["val_loss"])) + 1)
        if k == 0: hist0 = {kk: [float(v) for v in vv] for kk, vv in h.history.items()}
        if SAVE_WEIGHTS == "all" or (SAVE_WEIGHTS == "split0" and sp["seed"] == 0 and use_mha):
            model.save_weights(f"{OUT_DIR}/weights/{win}_{model_name}_s{sp['seed']}_{rep['name']}_fold{k}.weights.h5")
        params = model.count_params()
    t0 = time.time(); model(Z[te], training=False); lat = (time.time() - t0) / len(te) * 1000 * len(sp["folds"])
    oof_i, oof_p = np.concatenate(oof_i), np.concatenate(oof_p)
    res = dict(window=win, model=model_name, split=sp["seed"], ri=ri, rep=rep["name"], oof_i=oof_i, oof_p=oof_p,
               oof_auc=roc_auc_score(y[oof_i], oof_p), test_idx=te, prob=np.mean(test_p, 0),
               att=np.mean(att, 0) if use_mha else None, history=hist0, epochs_run=float(np.mean(ep)),
               best_epoch=float(np.mean(bep)), train_min=tmin, params=params, latency_ms_per_seq=lat,
               temporal_GMACs=temporal_macs(T, Z.shape[-1], use_mha) / 1e9)
    pickle.dump(res, open(path + ".tmp", "wb")); os.replace(path + ".tmp", path)   # atomic write
    keras.backend.clear_session(); gc.collect()
    return res

def run_one(win, T, model_name, sp):
    cands = [run_rep(win, T, model_name, sp, ri, rep) for ri, rep in enumerate(REPRESENTATIONS)]
    best = dict(max(cands, key=lambda r: r["oof_auc"]))                       # chosen on training data only
    best["thr"] = best_threshold(y[best["oof_i"]], best["oof_p"])
    best["oof_aucs"] = {c["rep"]: round(c["oof_auc"], 4) for c in cands}
    best.update(metrics(y[best["test_idx"]], best["prob"], best["thr"]))
    best.update({k + "_t05": v for k, v in metrics(y[best["test_idx"]], best["prob"], 0.5).items()})
    return best

In [ ]:
# ============================ 7. Run everything (resumable) ============================
RES, jobs = [], [(w, m, sp) for w in RUN_WINDOWS for m in ["LSTM-MHA", "LSTM"] for sp in splits]
t_start, n_new = time.time(), 0
for j, (win, model_name, sp) in enumerate(jobs, 1):
    T = min(WINDOWS[win], T_MAX); t0 = time.time()
    r = run_one(win, T, model_name, sp); RES.append(r)
    dt = time.time() - t0
    if dt > 2: n_new += 1
    eta = (time.time() - t_start) / max(n_new, 1) * (len(jobs) - j) / 60 if n_new else 0
    print(f"[{j}/{len(jobs)}] {win:5s} {model_name:8s} split {sp['seed']}: AUC={r['auc']:.3f}  acc={r['acc_t05']:.1f}  "
          f"input={r['rep']}  ({dt:.0f}s, ETA {eta:.0f} min)", flush=True)

In [ ]:
# ============================ 8. Logistic-regression baseline on identical splits and inner folds ============================
def oof_lr(F, sp, C):
    oi, op, tp = [], [], []; te = np.array(sp["test"])
    for itr, iva in sp["folds"]:
        itr, iva = np.array(itr), np.array(iva); mu, sd = F[itr].mean(0), F[itr].std(0) + 1e-6
        lr = LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit((F[itr] - mu) / sd, y[itr])
        oi.append(iva); op.append(lr.predict_proba((F[iva] - mu) / sd)[:, 1]); tp.append(lr.predict_proba((F[te] - mu) / sd)[:, 1])
    return np.concatenate(oi), np.concatenate(op), np.mean(tp, 0)

for win in RUN_WINDOWS:
    T = min(WINDOWS[win], T_MAX); Xw = X_raw[:, :T]; m = real_mask(Xw)
    last = Xw[np.arange(len(Xw)), np.maximum(m.sum(1) - 1, 0)]
    F = np.hstack([Xw.sum(1) / np.maximum(m.sum(1, keepdims=True), 1), last])          # mean frame + last frame
    for sp in splits:
        te = np.array(sp["test"]); best = None
        for C in [1e-4, 1e-3, 1e-2, 1e-1]:
            oi, op, tp = oof_lr(F, sp, C); v = roc_auc_score(y[oi], op)
            if best is None or v > best[0]: best = (v, oi, op, tp)
        _, oi, op, tp = best
        r_ = dict(window=win, model="LogReg", split=sp["seed"], test_idx=te, prob=tp, **metrics(y[te], tp, best_threshold(y[oi], op)))
        r_.update({k + "_t05": v for k, v in metrics(y[te], tp, 0.5).items()}); RES.append(r_)
print("done")

In [ ]:
# ============================ 9. Tables (LaTeX rows) ============================
cols = ["window", "model", "split", "auc", "acc_t05", "prec_non", "rec_non", "f1_non", "prec_bl", "rec_bl", "f1_bl",
        "thr", "oof_auc", "rep", "params", "temporal_GMACs", "epochs_run", "best_epoch", "train_min"]
df = pd.DataFrame([{c: r.get(c, np.nan) for c in cols} for r in RES])
df.to_csv(f"{OUT_DIR}/results_per_split.csv", index=False)
W_ = [w for w in WINDOWS if w in set(df.window)]; M_ = ["LSTM-MHA", "LSTM", "LogReg"]
num = df.drop(columns=["rep"]).groupby(["window", "model"]).agg(["mean", "std"])
num.reindex(pd.MultiIndex.from_product([W_, M_])).dropna(how="all").round(4).to_csv(f"{OUT_DIR}/summary_mean_sd.csv")
show = df.groupby(["window", "model"])[["auc", "acc_t05", "f1_bl", "f1_non"]].agg(lambda s: f"{s.mean():.3f} ± {s.std():.3f}")
print(show.reindex(pd.MultiIndex.from_product([W_, M_])).dropna().to_string())

def ms(w, m, k, d=3):
    s = df[(df.window == w) & (df.model == m)][k]; return f"{s.mean():.{d}f} $\\pm$ {s.std():.{d}f}"
task = {"W48": "Early prediction", "W72": "Early prediction", "W144": "End-of-culture assessment"}
lw = {"W48": "$W_{48}$ (Day 0--2)", "W72": "$W_{72}$ (Day 0--3)", "W144": "$W_{144}$ (Day 0--6)"}
L = ["% Table 2: AUC, accuracy at threshold 0.5"]
for w in W_: L.append(f"{lw[w]} & {task[w]} & {ms(w,'LSTM-MHA','auc')} & {ms(w,'LSTM-MHA','acc_t05',1)} \\\\")
L.append("% Table 3: AUC, F1 (Blast.), F1 (non-Blast.)")
for w in [w for w in ["W72", "W144"] if w in W_]:
    for m in M_: L.append(f"% {w} {m} & {ms(w,m,'auc')} & {ms(w,m,'f1_bl')} & {ms(w,m,'f1_non')} \\\\")
L.append("% Table 6 (ablation): precision non, recall non, accuracy at 0.5, AUC; Wilcoxon on per-split AUC")
stat = []
for w in W_:
    a = df[(df.window == w) & (df.model == "LSTM-MHA")].sort_values("split").auc.values
    b = df[(df.window == w) & (df.model == "LSTM")].sort_values("split").auc.values
    try: p = wilcoxon(a, b).pvalue
    except ValueError: p = np.nan
    stat.append(dict(window=w, dAUC=np.mean(a - b), p=p))
    for m in ["LSTM", "LSTM-MHA"]:
        L.append(f"% {w} {m} & {ms(w,m,'prec_non')} & {ms(w,m,'rec_non')} & {ms(w,m,'acc_t05',1)} & {ms(w,m,'auc')} \\\\")
    L.append(f"% {w}: Wilcoxon p={p:.2f}")
open(f"{OUT_DIR}/latex_rows.tex", "w").write("\n".join(L))
pd.DataFrame(stat).to_csv(f"{OUT_DIR}/wilcoxon.csv", index=False); print(pd.DataFrame(stat))

In [ ]:
# ============================ 10. Figures: ROC curves and attention scores ============================
import matplotlib.pyplot as plt
R = {(r["window"], r["model"], r["split"]): r for r in RES}; S = [sp["seed"] for sp in splits]

grid = np.linspace(0, 1, 201); fig, ax = plt.subplots(figsize=(5.2, 4.6))
for w in W_:
    tprs, aucs = [], []
    for s in S:
        r = R[(w, "LSTM-MHA", s)]; f, t, _ = roc_curve(y[r["test_idx"]], r["prob"])
        tprs.append(np.interp(grid, f, t)); aucs.append(r["auc"])
    mt, sd = np.mean(tprs, 0), np.std(tprs, 0)
    ax.plot(grid, mt, label=f"{w}: AUC {np.mean(aucs):.3f} ± {np.std(aucs, ddof=1):.3f}")
    ax.fill_between(grid, np.clip(mt - sd, 0, 1), np.clip(mt + sd, 0, 1), alpha=.12)
ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(fontsize=8, loc="lower right"); fig.tight_layout(); fig.savefig(f"{OUT_DIR}/ROC_rev.pdf"); plt.show()

rows = []                                         # only embryos whose recording covers the whole window
for (w, m, s), r in R.items():
    if m == "LSTM-MHA" and r.get("att") is not None:
        T = WINDOWS[w]
        for i, idx in enumerate(r["test_idx"]):
            if n_rows[idx] >= T:
                rows.append(dict(window=w, label=int(y[idx]), **{f"{FRAME_HOURS[j]}h": float(r["att"][i, j]) for j in range(T)}))
att = pd.DataFrame(rows); att.to_csv(f"{OUT_DIR}/attention_per_embryo.csv", index=False)
fig, ax = plt.subplots(1, len(W_), figsize=(4.4 * len(W_), 3.8), sharey=True, squeeze=False); ax = ax[0]
for k, w in enumerate(W_):
    d = att[att.window == w]; c = [f"{h}h" for h in FRAME_HOURS[:WINDOWS[w]]]; x = np.array(FRAME_HOURS[:WINDOWS[w]])
    for lab_, nm, off in [(1, "Blastocyst", -1.5), (0, "non-Blastocyst", 1.5)]:
        v = d[d.label == lab_][c]
        if len(v): ax[k].errorbar(x + off, v.mean(), yerr=v.std(), marker="o", capsize=3, ls=":", label=f"{nm} (n={len(v)})")
    ax[k].axhline(1 / len(c), c="gray", lw=.7, ls="--"); ax[k].set_xticks(x); ax[k].set_title(w)
    ax[k].set_xlabel("Frame time (h)"); ax[k].legend(fontsize=7)
ax[0].set_ylabel("Mean attention received")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/attention_weights_rev.pdf"); plt.show()
print("Figures saved to", OUT_DIR, "-> ROC_rev.pdf, attention_weights_rev.pdf")